# 28_01 혼동행렬 기반 분류 평가

In [1]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 851 ('font.family : Malgun Gothic')
Duplicate key in file WindowsPath('c:/Users/mzlap/Desktop/KNA-Data-analysis-1st/.venv/Lib/site-packages/matplotlib/mpl-data/matplotlibrc'), line 852 ('axes.unicode_minus : False')


✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


# 01 정확도와 그 한계
모델 채점의 첫 단추 — 정확도의 정의, 그리고 불균형이 만드는 함정


### accuracy_score()로 정확도 구하기
정답을 먼저, 예측을 나중에 — 앞으로 모든 지표 함수가 따르는 인자 순서

In [3]:
# 코드
from sklearn.metrics import accuracy_score

y_test = [1, 0, 0, 1, 0]
y_pred = [1, 0, 1, 1, 0]

acc = accuracy_score(y_test, y_pred)

print("정확도 : ", acc)

정확도 :  0.8


### 데이터 로드와 X·y 분리
CSV를 불러와 입력 X(센서 6종)와 정답 y(failure_soon)으로 분리


In [8]:
# 코드
import pandas as pd

df = pd.read_csv("28_cmapss_fd001_sample.csv")
df.head()
feature_cols = ["sensor_2",	"sensor_3",	"sensor_4",	"sensor_7",	"sensor_11",	"sensor_15"]

X = df[feature_cols]
y = df["failure_soon"]

### 분할·학습·예측
학습용·평가용으로 나눠 `RandomForest` 학습 후 예측 — 오늘 채점의 재료


In [ ]:
# 코드

# 1. 데이터를 학습용(train)과 테스트용(test)으로 나누기 위한 함수 불러오기
from sklearn.model_selection import train_test_split

# 2. 랜덤 포레스트 분류 모델 불러오기
#    → 여러 개의 결정트리(Decision Tree)를 만들어 다수결로 분류하는 모델
from sklearn.ensemble import RandomForestClassifier


# =====================================================
# Train / Test 나누기
# =====================================================
# 3. 전체 데이터를 학습용과 테스트용으로 나누기
Xtr, X_test, ytr, y_test = train_test_split(
    X,                  # 문제 데이터 / 입력값 / 독립변수 / feature
    y,                  # 정답 데이터 / 결과값 / 종속변수 / target
    test_size=0.3,      # 전체 데이터 중 30%를 테스트용으로 사용
                        # → 나머지 70%는 학습용 데이터가 됨
    random_state=42     # 실행할 때마다 똑같이 데이터를 나누기 위한 설정
)


# 나눈 결과
# Xtr     : 모델이 공부할 문제 데이터 (70%)
# ytr     : 모델이 공부할 문제의 정답 (70%)
# X_test  : 모델에게 시험으로 줄 문제 데이터 (30%)
# y_test  : 시험 문제의 실제 정답 (30%)

# 4. RandomForest 분류 모델을 만들고 학습시키기
# fit()
# → Xtr과 ytr을 이용해서
#   "어떤 X가 들어오면 어떤 y가 나오는지" 규칙을 학습

model = RandomForestClassifier(
    random_state=42     # 실행할 때마다 같은 결과가 나오도록 설정
).fit(
    Xtr,                # 학습할 문제
    ytr                 # 학습할 정답
)

# fit()의 의미
# → Xtr을 보고 ytr을 맞힐 수 있도록 모델이 규칙을 학습함


# 5. 학습이 끝난 모델에게 테스트 데이터를 주고 정답 예측하기
y_pred = model.predict(X_test)

# y_pred
# → 모델이 X_test를 보고 예상한 정답
#
# 이후 실제 정답인 y_test와
# 모델이 예측한 y_pred를 비교해서
# 모델이 얼마나 잘 맞혔는지 평가할 수 있음

y_pred[:15]


array([0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

### 손계산과 함수 대조
같은 자리의 개수를 직접 세어 전체로 나눈 값과 함수 결과 비교

In [ ]:
# 코드

# 1. 실제 정답(y_test)과 예측값(y_pred)이 같은 개수 세기
correct = (y_test == y_pred).sum()

# y_test == y_pred
# → 실제 정답과 예측값을 하나씩 비교
# → 같으면 True, 다르면 False


# 2. 전체 테스트 데이터 개수 구하기
total = len(y_test)

# len(y_test)
# → 시험 문제 전체 개수

print(f"직접 계산 : {correct} / {total}")



# 정확도를 계산해주는 함수 불러오기
from sklearn.metrics import accuracy_score

# sklearn의 accuracy_score 함수로 정확도 계산하기
# accuracy_score(실제정답, 예측값)
# → 맞힌 개수 / 전체 개수
print(f"함수 결과 : { accuracy_score(y_test, y_pred)}")


직접 계산 : 261 / 300
함수 결과 : 0.87


### 더미 분류기로 기준선 만들기
`most_frequent` 전략 — 항상 가장 많은 클래스로만 예측하는 모델

In [31]:
# 코드
# 1. 아주 단순한 기준 모델(DummyClassifier) 불러오기
from sklearn.dummy import DummyClassifier

# DummyClassifier
# → 똑똑하게 학습하는 모델이 아니라
#   "아주 단순한 방식"으로 예측하는 비교용 모델


# 2. 가장 많이 나온 정답만 계속 예측하는 모델 만들기
dummy = DummyClassifier(strategy="most_frequent")

# strategy="most_frequent"
# → 학습 데이터 ytr에서 가장 자주 나온 값을 찾고
#   모든 데이터를 그 값으로만 예측


# 3. 학습 데이터로 기준 모델 학습
dummy.fit(Xtr, ytr)
# 만들어진 모델 객체 확인
# dummy

# 여기서 중요한 점:
# DummyClassifier는 Xtr의 복잡한 패턴을 학습하는 게 아님
# 주로 ytr에서 어떤 값이 가장 많이 나오는지 확인하는 정도

# 4. 테스트 데이터에서 정확도 확인
print(f"베이스라인 : {dummy.score(X_test, y_test)}")

# .score(X_test, y_test)
# → 테스트 데이터에서 정확도(accuracy)를 계산

# 결과값 : 베이스라인 : 0.8
# 아무 생각 없이 가장 많은 클래스만 찍어도 80%는 맞힌다는 뜻


# # 실제 모델 정확도
# accuracy_score(y_test, y_pred)

# # 기준 모델 정확도
# dummy.score(X_test, y_test)


베이스라인 : 0.8


### 클래스 비율 확인
정상(0)과 고장 임박(1)의 개수·비율 확인 — 1이 얼마나 적은지 눈으로 확인


In [ ]:
# 코드
# =====================================================
# 정답 데이터가 불균형한지 확인
# =====================================================
# 1. y 안에 각 값이 몇 개씩 있는지 개수 확인
print(y.value_counts())


# 2. y 안에 각 값이 전체에서 차지하는 비율 확인
print(y.value_counts(normalize=True))


# 정상(0) : 80%
# 이상(1) : 20%

# 이런 상태를 클래스 불균형(class imbalance)이라고 함.



failure_soon
0    781
1    219
Name: count, dtype: int64
failure_soon
0    0.781
1    0.219
Name: proportion, dtype: float64


### 무조건 정상 예측의 정확도
모두 정상으로 찍어도 정확도가 꽤 높음 — 함정을 눈으로 확인하는 순간


In [ ]:
# 코드

# 1. numpy 불러오기
import numpy as np

# =====================================================
# 무조건 정상이라고 예측했을 때 정확도
# =====================================================
# 2. y_test와 같은 길이만큼 전부 0으로 채운 예측값 만들기
all_normal = np.zeros(len(y_test))

# np.zeros(개수)
# → 지정한 개수만큼 0을 만들어줌
#
# 예:
# len(y_test) = 5 라면
# all_normal = [0, 0, 0, 0, 0]
#
# 즉, 모든 테스트 데이터를 "정상(0)"이라고 예측한 것


# 3. 정확도 계산 함수 불러오기
from sklearn.metrics import accuracy_score


# 4. 실제 정답과 "전부 0으로 예측한 결과"를 비교해서 정확도 계산
print(f"무조건 정상 : {accuracy_score(y_test, all_normal)}")

# accuracy_score(y_test, all_normal)
# → 실제 정답 y_test와
#   전부 0으로 예측한 all_normal을 비교
#
# 예:
# y_test     = [0, 0, 1, 0, 1]
# all_normal = [0, 0, 0, 0, 0]
#
# 맞은 것      O  O  X  O  X
#
# 5개 중 3개 맞음
# 정확도 = 3 / 5 = 0.6

무조건 정상 : 0.8


### 두 모델 정확도 출력
더미와 모델의 정확도를 나란히 출력 — 두 숫자를 비교할 준비


In [ ]:
# 코드
# =====================================================
# DummyClassifier로 베이스라인 만들기
# =====================================================
# 1. DummyClassifier(베이스라인 모델)의 정확도 계산
dummy_acc = dummy.score(X_test, y_test)

# dummy.score(X_test, y_test)
# → 테스트 데이터에서 DummyClassifier가 얼마나 맞혔는지 계산
# → 앞에서 strategy="most_frequent"를 사용했으므로
#   가장 많이 나온 정답만 계속 예측한 정확도


# 2. 우리가 만든 RandomForest 모델의 정확도 계산
model_acc = model.score(X_test, y_test)


# model_acc = accuracy_score(y_test, y_pred)
# y_test : 실제 정답
# y_pred : RandomForest가 예측한 정답
#
# accuracy_score()
# → 실제 정답과 예측값을 비교해서 정확도 계산


# 3. 베이스라인 정확도 출력
print(f"베이스라인 : {dummy_acc}")


# 4. RandomForest 모델 정확도 출력
print(f"우리 모델 : {model_acc}")


#==============================================================================
# AI한테 왜 80%라고 하는 베이스라인하고 큰 차이가 없는지에 대해 설명해봐 라고 하면
# 기준을 뭘로 정했는지 답할 수 없는 AI가 없음.
# 추정을 통해 보여주기 때문에 AI를 맹신하지 말 것.


베이스라인 : 0.8
우리 모델 : 0.87


### 차이 계산과 판단
두 정확도의 차이를 퍼센트포인트로 — 1~2%p면 학습 거의 안 됨, 10%p 이상이면 의미 있음


In [ ]:
# 코드

gap = model_acc - dummy_acc

print(f"차이(%) : {round(gap *100, 2)} %")

# 이 차이가 크면 모델이 베이스라인을 넘어 진짜 학습한 것

차이(%) : 7.0 %


# 02 혼동행렬
네 칸짜리 성적표 — TP·FN·FP·TN과 설비 비용의 언어


### confusion_matrix()와 출력 순서 주의
ravel로 펼치면 순서대로 tn, fp, fn, tp — 이 순서를 헷갈리면 모든 지표가 엉터리


In [ ]:
# 코드

### 혼동행렬 시각화 (heatmap)
ConfusionMatrixDisplay로 색칠된 표 생성 — 진한 칸이 많은 칸


In [ ]:
# 코드

### 예시로 네 칸 채우기
여섯 대를 한 칸씩 분류 → TP=2, FN=1, FP=1, TN=2 — 함수 결과와 일치 확인


In [ ]:
# 코드

### 혼동행렬 생성
우리 모델 예측으로 혼동행렬 출력 — 2×2 배열 확인

In [ ]:
# 코드

### heatmap 시각화
진한 칸이 어디인지 눈으로 확인 — FN 칸이 진하면 놓침 많다는 신호

In [ ]:
# 코드

### 네 칸 추출
ravel로 네 칸을 각 변수에 담기 — 순서 tn, fp, fn, tp 엄수


In [ ]:
# 코드

### 설비 대수로 해석 출력
각 칸을 현장 언어 문장으로 — 숫자가 현장의 의미로 바뀌는 순간


In [ ]:
# 코드

# 03 정밀도와 재현율
경보의 신뢰도와 놓치지 않는 능력 — 두 지표의 짝과 트레이드오프


### 네 칸으로 손계산
공식에 네 칸을 직접 대입 — 정밀도 = TP/(TP+FP), 재현율 = TP/(TP+FN)


In [ ]:
# 코드

### 함수와 대조
손계산과 함수 결과 비교 — 일치하면 공식을 완전히 이해


In [ ]:
# 코드

### 세 지표 한 화면 출력
정확도·정밀도·재현율을 함께 출력 — 정확도 높은데 재현율 낮은 식의 치우침 확인


In [ ]:
# 코드